# Section 노드 및 기업 관계 적재

기존 `section_insert.ipynb`는 변경하지 않고 별도로 작성한 실행용 노트북입니다.

## 생성 구조

| 시작 노드 | 관계 | 종료 노드 |
|---|---|---|
| `ParentCompany` | `IN_INDUSTRY` | `Section` |
| `SubsidiaryCompany` | `IN_INDUSTRY` | `Section` |

기존 `Company → Industry` 관계의 `Industry.name`에 소분류 키워드가 포함되는지를 기준으로 `Section`을 결정합니다. 여러 대분류와 매칭되면 여러 `Section`에 연결하고, 하나도 매칭되지 않으면 `모름`에 연결합니다. `기타`와 `-`는 오분류를 막기 위해 분류 키워드로 사용하지 않습니다.

> 기본값은 `APPLY_CHANGES = False`입니다. 미리보기 결과를 확인한 뒤 마지막 실행 셀에서 값을 `True`로 바꿔야 실제 DB에 반영됩니다.

In [1]:
SECTION_ROWS = [
    {"name": "금융", "category": ["SPC", "펀드", "지주", "은행", "증권", "보험"]},
    {"name": "제조", "category": ["전자", "자동차", "화학", "소재", "기계"]},
    {"name": "IT·미디어", "category": ["소프트웨어", "통신", "게임", "방송", "콘텐츠"]},
    {"name": "부동산·건설", "category": ["부동산", "임대", "건설", "시공"]},
    {"name": "서비스", "category": ["호텔", "교육", "컨설팅", "연구", "정비"]},
    {"name": "유통·물류", "category": ["도소매", "무역", "운송", "창고"]},
    {"name": "바이오·헬스케어", "category": ["제약", "의료", "화장품"]},
    {"name": "에너지·환경", "category": ["발전", "태양광", "폐기물"]},
    {"name": "식품·농업", "category": ["식품", "외식", "농축산"]},
    {"name": "모름", "category": []},
]


def classify_industry(industry_name):
    normalized_name = (industry_name or "").casefold()
    matches = [
        section["name"]
        for section in SECTION_ROWS
        if section["name"] != "모름"
        and any(keyword.casefold() in normalized_name for keyword in section["category"])
    ]
    return matches or ["모름"]

In [2]:
import json
from pathlib import Path


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp")
    try:
        with temporary_path.open("w", encoding="utf-8", newline="\n") as file:
            for row in rows:
                file.write(json.dumps(row, ensure_ascii=False) + "\n")
        temporary_path.replace(path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def build_section_nodes(section_rows=SECTION_ROWS):
    return [
        {
            "id": f"section:{section['name']}",
            "type": "Section",
            "properties": {
                "name": section["name"],
                "category": list(section["category"]),
            },
        }
        for section in section_rows
    ]


def build_section_triples(assignments, section_rows=SECTION_ROWS):
    section_order = {row["name"]: index for index, row in enumerate(section_rows)}
    section_triples = []
    for assignment in sorted(
        assignments, key=lambda row: (row["subject_type"], row["subject"])
    ):
        subject = assignment.get("subject")
        subject_type = assignment.get("subject_type")
        if not subject:
            raise ValueError("Neo4j Company 노드에 id가 없습니다.")
        if subject_type not in {"ParentCompany", "SubsidiaryCompany"}:
            raise ValueError(f"허용되지 않은 회사 타입: {subject_type}")
        industry_names = sorted(
            {name for name in assignment.get("industry_names", []) if name}
        )
        matched_sections = set()
        for industry_name in industry_names:
            matched_sections.update(classify_industry(industry_name))
        matched_sections.discard("모름")
        if not matched_sections:
            matched_sections = {"모름"}

        industry_text = ", ".join(industry_names)
        for section_name in sorted(matched_sections, key=section_order.get):
            section_triples.append(
                {
                    "subject": subject,
                    "subject_type": subject_type,
                    "relation": "IN_INDUSTRY",
                    "object": f"section:{section_name}",
                    "object_type": "Section",
                    "source_case": "section_insert_v3",
                    "source_row": assignment.get("source_row"),
                    "evidence": industry_text,
                }
            )
    return section_triples


def validate_section_overlay(section_nodes, section_triples):
    section_ids = {node["id"] for node in section_nodes}
    if len(section_ids) != len(section_nodes):
        raise ValueError("중복된 노드 id가 있습니다.")
    seen = set()
    for triple in section_triples:
        object_id = triple.get("object")
        if object_id not in section_ids:
            raise ValueError(f"Section object를 찾을 수 없습니다: {object_id}")
        if triple.get("object_type") != "Section":
            raise ValueError(f"object_type 불일치: {object_id}")
        if triple.get("subject_type") not in {"ParentCompany", "SubsidiaryCompany"}:
            raise ValueError(f"subject_type 불일치: {triple.get('subject')}")
        key = (triple.get("subject"), triple.get("relation"), object_id)
        if key in seen:
            raise ValueError(f"중복된 트리플: {key}")
        seen.add(key)
    return {
        "section_node_count": len(section_nodes),
        "section_triple_count": len(section_triples),
    }

## 별도 v3 JSONL 출력 경로

기존 JSONL은 읽거나 병합하지 않습니다. 현재 Neo4j의 `Company → Industry` 관계를 읽어 Section 전용 노드와 트리플을 각각 `기업관계_노드_v3.jsonl`, `기업관계_트리플_v3.jsonl`로 저장합니다.

In [3]:
import pandas as pd

jsonl_start = Path.cwd().resolve()
jsonl_project_root = next(
    (candidate for candidate in (jsonl_start, *jsonl_start.parents)
     if (candidate / "pyproject.toml").is_file()),
    None,
)
if jsonl_project_root is None:
    raise FileNotFoundError("pyproject.toml이 있는 프로젝트 루트를 찾지 못했습니다.")

clean_dir = jsonl_project_root / "data" / "clean"
output_node_path = clean_dir / "기업관계_노드_v3.jsonl"
output_triple_path = clean_dir / "기업관계_트리플_v3.jsonl"
print(f"노드 출력 예정 경로: {output_node_path}")
print(f"트리플 출력 예정 경로: {output_triple_path}")

노드 출력 예정 경로: C:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\data\clean\기업관계_노드_v3.jsonl
트리플 출력 예정 경로: C:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\data\clean\기업관계_트리플_v3.jsonl


## Neo4j 연결

프로젝트 루트의 `.env`에서 `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`를 읽습니다. 값이 없으면 기본 계정으로 접속하지 않고 즉시 중단합니다. `NEO4J_DATABASE`는 선택 항목이며 기본값은 `neo4j`입니다.

In [4]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("pyproject.toml이 있는 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
ENV_PATH = PROJECT_ROOT / ".env"
if not ENV_PATH.is_file():
    raise FileNotFoundError(f"환경변수 파일이 없습니다: {ENV_PATH}")

load_dotenv(ENV_PATH)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

missing = [
    name
    for name, value in {
        "NEO4J_URI": NEO4J_URI,
        "NEO4J_USER": NEO4J_USER,
        "NEO4J_PASSWORD": NEO4J_PASSWORD,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"필수 환경변수가 없습니다: {', '.join(missing)}")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print(f"Neo4j 연결 완료: database={NEO4J_DATABASE}")


def run_cypher(query, **params):
    with driver.session(database=NEO4J_DATABASE) as session:
        return session.run(query, **params).data()

Neo4j 연결 완료: database=neo4j


In [5]:
SECTION_CONSTRAINT_QUERY = """
CREATE CONSTRAINT section_name_unique IF NOT EXISTS
FOR (section:Section)
REQUIRE section.name IS UNIQUE
"""

SECTION_NODE_QUERY = """
UNWIND $sections AS row
MERGE (section:Section {name: row.name})
SET section.id = 'section:' + row.name
SET section.category = row.category
RETURN count(section) AS processed_sections
"""

INDUSTRY_MAPPING_PREVIEW_QUERY = """
MATCH (industry:Industry)
WITH industry,
     [section IN $sections
      WHERE section.name <> '모름'
        AND any(keyword IN section.category
                WHERE toLower(coalesce(industry.name, '')) CONTAINS toLower(keyword))
      | section.name] AS matched_sections
RETURN industry.name AS industry_name,
       CASE WHEN size(matched_sections) = 0 THEN ['모름']
            ELSE matched_sections END AS sections,
       size(matched_sections) AS match_count
ORDER BY match_count, industry_name
"""

COMPANY_INDUSTRY_SOURCE_QUERY = """
MATCH (company)-[relation:IN_INDUSTRY]->(industry:Industry)
WHERE company:ParentCompany OR company:SubsidiaryCompany
RETURN company.id AS subject,
       CASE WHEN company:ParentCompany
            THEN 'ParentCompany'
            ELSE 'SubsidiaryCompany' END AS subject_type,
       collect(DISTINCT industry.name) AS industry_names,
       min(relation.source_row) AS source_row
ORDER BY subject_type, subject
"""

PARENT_SECTION_RELATION_QUERY = """
MATCH (company:ParentCompany)-[:IN_INDUSTRY]->(industry:Industry)
MATCH (section:Section)
WHERE section.name <> '모름'
  AND any(keyword IN section.category
          WHERE toLower(coalesce(industry.name, '')) CONTAINS toLower(keyword))
WITH DISTINCT company, section
MERGE (company)-[r:IN_INDUSTRY]->(section)
ON CREATE SET r.source_case = 'section_insert_v3',
              r.evidence = 'Industry.name 소분류 키워드 매칭'
RETURN count(r) AS processed_relationships
"""

SUBSIDIARY_SECTION_RELATION_QUERY = """
MATCH (company:SubsidiaryCompany)-[:IN_INDUSTRY]->(industry:Industry)
MATCH (section:Section)
WHERE section.name <> '모름'
  AND any(keyword IN section.category
          WHERE toLower(coalesce(industry.name, '')) CONTAINS toLower(keyword))
WITH DISTINCT company, section
MERGE (company)-[r:IN_INDUSTRY]->(section)
ON CREATE SET r.source_case = 'section_insert_v3',
              r.evidence = 'Industry.name 소분류 키워드 매칭'
RETURN count(r) AS processed_relationships
"""

PARENT_UNKNOWN_RELATION_QUERY = """
MATCH (company:ParentCompany)
WHERE EXISTS { MATCH (company)-[:IN_INDUSTRY]->(:Industry) }
  AND NOT EXISTS {
      MATCH (company)-[:IN_INDUSTRY]->(section:Section)
      WHERE section.name <> '모름'
  }
MATCH (unknown:Section {name: '모름'})
MERGE (company)-[r:IN_INDUSTRY]->(unknown)
ON CREATE SET r.source_case = 'section_insert_v3',
              r.evidence = '대분류 키워드 미매칭'
RETURN count(r) AS processed_relationships
"""

SUBSIDIARY_UNKNOWN_RELATION_QUERY = """
MATCH (company:SubsidiaryCompany)
WHERE EXISTS { MATCH (company)-[:IN_INDUSTRY]->(:Industry) }
  AND NOT EXISTS {
      MATCH (company)-[:IN_INDUSTRY]->(section:Section)
      WHERE section.name <> '모름'
  }
MATCH (unknown:Section {name: '모름'})
MERGE (company)-[r:IN_INDUSTRY]->(unknown)
ON CREATE SET r.source_case = 'section_insert_v3',
              r.evidence = '대분류 키워드 미매칭'
RETURN count(r) AS processed_relationships
"""

VERIFY_QUERY = """
CALL { MATCH (:Section) RETURN count(*) AS section_count }
CALL {
    MATCH (:ParentCompany)-[r:IN_INDUSTRY]->(:Section)
    RETURN count(r) AS parent_section_relationships
}
CALL {
    MATCH (:SubsidiaryCompany)-[r:IN_INDUSTRY]->(:Section)
    RETURN count(r) AS subsidiary_section_relationships
}
RETURN section_count,
       parent_section_relationships,
       subsidiary_section_relationships
"""

## 적재 전 분류 결과 확인

이 셀은 읽기 전용입니다. `match_count=0`은 `모름`, `match_count>1`은 여러 Section에 연결될 업종입니다.

In [6]:
preview_rows = run_cypher(
    INDUSTRY_MAPPING_PREVIEW_QUERY,
    sections=SECTION_ROWS,
)
preview_df = pd.DataFrame(preview_rows)
summary_df = pd.DataFrame(
    [
        {"구분": "전체 Industry", "개수": len(preview_df)},
        {"구분": "미분류(모름)", "개수": int((preview_df["match_count"] == 0).sum())},
        {"구분": "단일 Section", "개수": int((preview_df["match_count"] == 1).sum())},
        {"구분": "복수 Section", "개수": int((preview_df["match_count"] > 1).sum())},
    ]
)
display(summary_df)
display(preview_df[preview_df["match_count"] != 1].head(100))

# 현재 Neo4j의 Company → Industry 관계에서 Section 전용 JSONL을 생성합니다.
company_industry_assignments = run_cypher(COMPANY_INDUSTRY_SOURCE_QUERY)
section_nodes = build_section_nodes()
section_triples = build_section_triples(company_industry_assignments)
validation_summary = validate_section_overlay(section_nodes, section_triples)
write_jsonl(output_node_path, section_nodes)
write_jsonl(output_triple_path, section_triples)

jsonl_summary_df = pd.DataFrame(
    [
        {"항목": "Section 노드", "개수": validation_summary["section_node_count"]},
        {"항목": "Company→Section 트리플", "개수": validation_summary["section_triple_count"]},
    ]
)
display(jsonl_summary_df)
print(f"노드 출력: {output_node_path}")
print(f"트리플 출력: {output_triple_path}")

,구분,개수
0,전체 Industry,455
1,미분류(모름),241
2,단일 Section,162
3,복수 Section,52


,industry_name,sections,match_count
0,1차 금속 제조업,[모름],0
1,1차 금속제품 도매업,[모름],0
2,1차 철강 제조업,[모름],0
3,IT서비스,[모름],0
4,가구 소매업,[모름],0
...,...,...,...
95,모피 및 가죽 제조업,[모름],0
96,무기안료용 금속 산화물 및 관련 제품 제조업,[모름],0
97,무점포 소매업,[모름],0
98,문구용품_ 회화용품_ 사무용품 도매업,[모름],0


,항목,개수
0,Section 노드,10
1,Company→Section 트리플,1531


노드 출력: C:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\data\clean\기업관계_노드_v3.jsonl
트리플 출력: C:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\data\clean\기업관계_트리플_v3.jsonl


## 실제 적재 및 결과 검증

미리보기 결과가 적절할 때만 `APPLY_CHANGES = True`로 변경합니다. `MERGE`와 `Section.name` 고유 제약조건을 사용하므로 같은 데이터로 재실행해도 노드와 관계가 중복 생성되지 않습니다.

In [7]:
APPLY_CHANGES = False

if not APPLY_CHANGES:
    print("읽기 전용 미리보기만 완료했습니다. 실제 적재하려면 APPLY_CHANGES를 True로 변경하세요.")
else:
    try:
        run_cypher(SECTION_CONSTRAINT_QUERY)
        print(run_cypher(SECTION_NODE_QUERY, sections=SECTION_ROWS))
        print("ParentCompany → Section:", run_cypher(PARENT_SECTION_RELATION_QUERY))
        print("SubsidiaryCompany → Section:", run_cypher(SUBSIDIARY_SECTION_RELATION_QUERY))
        print("ParentCompany → 모름:", run_cypher(PARENT_UNKNOWN_RELATION_QUERY))
        print("SubsidiaryCompany → 모름:", run_cypher(SUBSIDIARY_UNKNOWN_RELATION_QUERY))
        display(pd.DataFrame(run_cypher(VERIFY_QUERY)))
    finally:
        driver.close()
        print("Neo4j 연결을 종료했습니다.")

읽기 전용 미리보기만 완료했습니다. 실제 적재하려면 APPLY_CHANGES를 True로 변경하세요.


## GraphRAG 연동 시 주의사항

이 노트북은 기존 파일을 변경하지 않습니다. 따라서 `src/agent/tools/graph_ontology.json`도 그대로이며, GraphRAG 에이전트가 `Section`을 자동 활용하게 하려면 별도 작업으로 다음 서명을 온톨로지에 추가해야 합니다.

- `ParentCompany -[IN_INDUSTRY]-> Section`
- `SubsidiaryCompany -[IN_INDUSTRY]-> Section`